Model/vectorization experimentation continuation...

In [4]:
import pandas as pd
from pathlib import Path

In [7]:
# Data import:
PROJ_ROOT = Path().resolve().parents[0]
DATA_DIR = PROJ_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
RAW_DATA_PATH = RAW_DATA_DIR / "potential-talents - Aspiring human resources - seeking human resources.csv"
data = pd.read_csv(RAW_DATA_PATH)

In [8]:
# Data pre-processing:
data = data.drop_duplicates(subset=["job_title", "location", "connection", "fit"]).reset_index(drop=True) #All id's are unique, but the info in the other columns (when combined), aren't
data["id"] = data.index + 1
data["combined"] = (data["job_title"].astype(str)+ " located in " + data["location"].astype(str) + ", with " + data["connection"].astype(str)+ " connections.")

In [9]:
ideal_candidate_description = "Experienced in Human Resources"

<h4>Bag Of Words

Bag of Words (BoW) is a simpler predecessor to TF-IDF — same basic idea, but instead of weighting terms by how distinctive they are across the corpus, it just counts raw term frequency

In [13]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [16]:
def rank_candidates_bow(df, query, top_n):
    """
    Rank all candidates in *df* by Bag-of-Words cosine similarity to *query*.
    """
    if df.empty or not query.strip():
        df = df.copy()
        df["fit"] = 0.0
        return df

    corpus = df["combined"].fillna("").tolist()
    vectorizer = CountVectorizer(
        ngram_range=(1, 2),
        stop_words="english",
        min_df=1,
    )
    all_docs = corpus + [query]
    bow_matrix = vectorizer.fit_transform(all_docs)

    query_vec = bow_matrix[-1]
    candidate_matrix = bow_matrix[:-1]

    scores = cosine_similarity(query_vec, candidate_matrix).flatten()

    result = df.copy()
    result["fit"] = scores
    result = result.sort_values("fit", ascending=False)

    if top_n:
        result = result.head(top_n)

    return result.reset_index(drop=True)

In [18]:
bow_ideal = rank_candidates_bow(data, ideal_candidate_description, 10) 
bow_ideal.head(10) #TF-IDF

,id,job_title,location,connection,fit,combined
0,22,"Aspiring Human Resources Manager, seeking inte...","Houston, Texas Area",7,0.481932,"Aspiring Human Resources Manager, seeking inte..."
1,49,Aspiring Human Resources Manager | Graduating ...,"Cape Girardeau, Missouri",103,0.400000,Aspiring Human Resources Manager | Graduating ...
2,15,Experienced Retail Manager and aspiring Human ...,"Austin, Texas Area",57,0.357771,Experienced Retail Manager and aspiring Human ...
3,14,Seeking Human Resources Opportunities,"Chicago, Illinois",390,0.325396,Seeking Human Resources Opportunities located ...
4,37,Human Resources Management Major,"Milpitas, California",18,0.325396,Human Resources Management Major located in Mi...
5,23,Human Resources Professional,Greater Boston Area,16,0.325396,Human Resources Professional located in Greate...
6,27,Human Resources Generalist at Schwan's,Amerika Birleşik Devletleri,500+,0.307794,Human Resources Generalist at Schwan's located...
7,46,Aspiring Human Resources Professional,"Kokomo, Indiana Area",71,0.307794,Aspiring Human Resources Professional located ...
8,38,Director Human Resources at EY,Greater Atlanta Area,349,0.307794,Director Human Resources at EY located in Gre...
9,48,Seeking Human Resources Position,"Las Vegas, Nevada Area",48,0.292770,Seeking Human Resources Position located in La...


BoW is similar to TF-IDF vectorization. There doesn't appear to be any major improvement shown. It actually appears to be a downgrade here..

<h4>Word2Vec

Should add semantics as a feature compared to TF-IDF

In [19]:
import gensim.downloader as api
import numpy as np

In [20]:
w2v_model = api.load("word2vec-google-news-300")

In [21]:
def embed_text_w2v(text: str, model) -> np.ndarray:
    """Average word vectors for a piece of text into a single embedding."""
    words = text.lower().split()
    vectors = [model[w] for w in words if w in model.key_to_index]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

def rank_candidates_w2v(df: pd.DataFrame, query: str, model, top_n: int | None = None) -> pd.DataFrame:
    if df.empty or not query.strip():
        df = df.copy()
        df["fit"] = 0.0
        return df

    query_vec = embed_text_w2v(query, model).reshape(1, -1)
    candidate_vecs = np.vstack([embed_text_w2v(t, model) for t in df["combined"].fillna("")])

    scores = cosine_similarity(query_vec, candidate_vecs).flatten()

    result = df.copy()
    result["fit"] = scores
    result = result.sort_values("fit", ascending=False)

    if top_n:
        result = result.head(top_n)

    return result.reset_index(drop=True)

In [24]:
w2v_ideal = rank_candidates_w2v(data, ideal_candidate_description, w2v_model, 20)
w2v_ideal.head(20)

,id,job_title,location,connection,fit,combined
0,27,Human Resources Generalist at Schwan's,Amerika Birleşik Devletleri,500+,0.780672,Human Resources Generalist at Schwan's located...
1,37,Human Resources Management Major,"Milpitas, California",18,0.779978,Human Resources Management Major located in Mi...
2,49,Aspiring Human Resources Manager | Graduating ...,"Cape Girardeau, Missouri",103,0.749731,Aspiring Human Resources Manager | Graduating ...
3,14,Seeking Human Resources Opportunities,"Chicago, Illinois",390,0.743255,Seeking Human Resources Opportunities located ...
4,15,Experienced Retail Manager and aspiring Human ...,"Austin, Texas Area",57,0.735117,Experienced Retail Manager and aspiring Human ...
5,23,Human Resources Professional,Greater Boston Area,16,0.718022,Human Resources Professional located in Greate...
6,9,Seeking Human Resources HRIS and Generalist Po...,Greater Philadelphia Area,500+,0.705368,Seeking Human Resources HRIS and Generalist Po...
7,46,Aspiring Human Resources Professional,"Kokomo, Indiana Area",71,0.696865,Aspiring Human Resources Professional located ...
8,33,Human Resources professional for the world lea...,"Highland, California",50,0.691257,Human Resources professional for the world lea...
9,22,"Aspiring Human Resources Manager, seeking inte...","Houston, Texas Area",7,0.689530,"Aspiring Human Resources Manager, seeking inte..."


It's able to pick out candidates (with a high similarity score) that are semantically similar to the ideal candidate but it's still not able to make the 'experienced' distinction as eluded to in the previous notebook.